# Thunder and Pacers Shot Data (Finals)

In [29]:
from nba_api.stats.endpoints import shotchartdetail
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import commonplayerinfo
from nba_api.stats.static import players
import pandas as pd

In [30]:
import time, random
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def nba_session():
    s = requests.Session()

    # These headers matter for stats.nba.com
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Origin": "https://www.nba.com",
        "Referer": "https://www.nba.com/",
        "Connection": "keep-alive",
    })

    retry = Retry(
        total=8,
        connect=8,
        read=8,
        backoff_factor=1.2,              # exponential-ish backoff
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
        raise_on_status=False,
        respect_retry_after_header=True,
    )

    adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

SESSION = nba_session()

def polite_sleep(min_s=0.6, max_s=1.6):
    time.sleep(random.uniform(min_s, max_s))


In [31]:
import pandas as pd
from nba_api.stats.endpoints import leaguegamefinder

def get_finals_game_ids_for_player(player_id: int, season: str) -> list[str]:
    """
    Returns Finals game IDs for a player in a given season (e.g., '2016-17').
    Finals games have GAME_ID containing '040' in the series code.
    """
    gf = leaguegamefinder.LeagueGameFinder(
        player_id_nullable=player_id,
        season_nullable=season,
        season_type_nullable="Playoffs"
    )

    games = gf.get_data_frames()[0]

    # Finals filter: Finals series games have '040' in the GAME_ID (e.g., 0041600401)
    games["GAME_ID"] = games["GAME_ID"].astype(str)
    finals_games = games[games["GAME_ID"].str.contains("040")].copy()

    return finals_games["GAME_ID"].tolist()

In [32]:
from nba_api.stats.endpoints import shotchartdetail

def get_player_shots_for_game(player_id: int, season: str, game_id: str) -> pd.DataFrame:
    scd = shotchartdetail.ShotChartDetail(
        team_id=0,
        player_id=player_id,
        season_nullable=season,
        season_type_all_star="Playoffs",
        game_id_nullable=game_id,
        context_measure_simple="FGA"
    )
    df = scd.get_data_frames()[0]
    return df

In [33]:
import time

def finals_shots_last_10_seasons(player_name: str, start_year=2024, end_year=2025):
    """
    Pulls ALL shots taken by player in ALL NBA Finals games from seasons start_year-end_year.
    Seasons are strings like '2016-17'. end_year refers to the ending year of the season.
    """
    from nba_api.stats.static import players

    # find player_id
    p = players.find_players_by_full_name(player_name)[0]
    player_id = p["id"]

    all_shots = []
    summary = []

    for end in range(start_year + 1, end_year + 1):
        season = f"{end-1}-{str(end)[-2:]}"

        finals_game_ids = get_finals_game_ids_for_player(player_id, season)
        summary.append({"season": season, "finals_games_found": len(finals_game_ids)})

        for gid in finals_game_ids:
            try:
                df_game = get_player_shots_for_game(player_id, season, gid)
                if not df_game.empty:
                    all_shots.append(df_game)
            except Exception as e:
                print(f"Failed game {gid}: {e}")

            time.sleep(0.8)

    shots_df = pd.concat(all_shots, ignore_index=True) if all_shots else pd.DataFrame()
    summary_df = pd.DataFrame(summary)

    return shots_df, summary_df


In [34]:
shai_df, shai_summary = finals_shots_last_10_seasons(
    "Shai Gilgeous-Alexander",
    2024,
    2025
)

print("SHAI:", len(shai_df))

SHAI: 158


In [35]:
chet_df, chet_summary = finals_shots_last_10_seasons(
    "Chet Holmgren",
    2024,
    2025
)

print("CHET:", len(chet_df))

CHET: 76


In [36]:
jdub_df, jdub_summary = finals_shots_last_10_seasons(
    "Jalen Williams",
    2024,
    2025
)

print("JDUB:", len(jdub_df))

JDUB: 127


In [37]:
dort_df, dort_summary = finals_shots_last_10_seasons(
    "Luguentz Dort",
    2024,
    2025
)

print("DORT:", len(dort_df))

DORT: 45


In [38]:
joe_df, joe_summary = finals_shots_last_10_seasons(
    "Isaiah Joe",
    2024,
    2025
)

print("JOE:", len(joe_df))

JOE: 13


In [39]:
wiggins_df, wiggins_summary = finals_shots_last_10_seasons(
    "Aaron Wiggins",
    2024,
    2025
)

print("WIGGINS:", len(wiggins_df))

WIGGINS: 33


In [40]:
wallace_df, wallace_summary = finals_shots_last_10_seasons(
    "Cason Wallace",
    2024,
    2025
)

print("WALLACE:", len(wallace_df))

WALLACE: 38


In [41]:
jaylin_df, jaylin_summary = finals_shots_last_10_seasons(
    "Jaylin Williams",
    2024,
    2025
)

print("JAYLIN:", len(jaylin_df))

JAYLIN: 4


In [42]:
kenrich_df, kenrich_summary = finals_shots_last_10_seasons(
    "Kenrich Williams",
    2024,
    2025
)

print("KENRICH:", len(kenrich_df))

KENRICH: 10


In [46]:
combined_df = pd.concat([
    shai_df,
    chet_df,
    jdub_df,
    dort_df,
    joe_df,
    wiggins_df,
    wallace_df,
    jaylin_df,
    kenrich_df,
], ignore_index=True)

In [47]:
combined_df.to_csv(
    "Thunder_Finals_Shots_2025.csv",
    index=False
)

print("Saved Thunder_Finals_Shots_2025.csv")

Saved Thunder_Finals_Shots_2025.csv
